# Question Predictor — 10-Year Paper Data

In [ ]:
# Install dependencies (run once)
# !pip install pandas scikit-learn transformers torch datasets

## 1. Load Data

In [ ]:
import pandas as pd

# Expected CSV columns: 'year', 'subject', 'topic', 'question'
df = pd.read_csv('data/papers.csv')
print(df.shape)
df.head()

## 2. Preprocess

In [ ]:
from sklearn.model_selection import train_test_split

# Combine context features into a single input text
df['input_text'] = df['subject'] + ' | ' + df['topic'] + ' | ' + df['year'].astype(str)
df = df.dropna(subset=['input_text', 'question'])

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

## 3. Tokenize

In [ ]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained('t5-small')

def tokenize(texts, targets, max_len=128):
    inputs = tokenizer(texts, max_length=max_len, truncation=True, padding='max_length', return_tensors='pt')
    labels = tokenizer(targets, max_length=max_len, truncation=True, padding='max_length', return_tensors='pt').input_ids
    return inputs, labels

train_inputs, train_labels = tokenize(train_df['input_text'].tolist(), train_df['question'].tolist())
test_inputs,  test_labels  = tokenize(test_df['input_text'].tolist(),  test_df['question'].tolist())

## 4. Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PaperDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            'input_ids':      self.inputs.input_ids[idx],
            'attention_mask': self.inputs.attention_mask[idx],
            'labels':         self.labels[idx]
        }

train_loader = DataLoader(PaperDataset(train_inputs, train_labels), batch_size=8, shuffle=True)
test_loader  = DataLoader(PaperDataset(test_inputs,  test_labels),  batch_size=8)

## 5. Model

In [ ]:
from transformers import T5ForConditionalGeneration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)
print(f'Using device: {device}')

## 6. Train

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        loss = model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            labels         = batch['labels'].to(device)
        ).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}')

## 7. Evaluate & Predict

In [ ]:
model.eval()

def predict(subject, topic, year):
    text   = f'{subject} | {topic} | {year}'
    inputs = tokenizer(text, return_tensors='pt').to(device)
    output = model.generate(**inputs, max_length=128)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example
print(predict('Mathematics', 'Calculus', 2024))

## 8. Save Model

In [ ]:
model.save_pretrained('saved_model/')
tokenizer.save_pretrained('saved_model/')
print('Model saved.')